In [27]:
import re
import ast
import pandas as pd

## For total acccuracy

In [28]:
def parse_run_string(s):
    # From CHAT: Replace np.float64(0.123) -> 0.123
    s = re.sub(r'np\.float64\(([^()]*)\)', r'\1', s)
    return ast.literal_eval(s)

def extract_final_accuracy(s):
    data = parse_run_string(s)
    return float(data[-1][1])

def extract_auc(s):
    data = parse_run_string(s)
    ys = [float(acc) for _, acc in data]
    return sum(ys) / len(ys)

In [29]:
def clean_total_data(df):
    df_clean = pd.DataFrame({
        "random": df["random"].apply(extract_final_accuracy),
        "margin": df["margin"].apply(extract_final_accuracy),
        "least_conf": df["least_confident"].apply(extract_final_accuracy),
        "entropy": df["entropy"].apply(extract_final_accuracy),
    })
    return df_clean

In [30]:
df_et = clean_total_data(pd.read_csv("results/even_total_accuracy.csv"))
df_ut = clean_total_data(pd.read_csv("results/uneven_total_accuracy.csv"))

In [31]:
df_et.head()

,random,margin,least_conf,entropy
0,0.451638,0.417577,0.278471,0.258710
1,0.457358,0.433177,0.313313,0.297192
2,0.437337,0.426417,0.312793,0.269371
3,0.457098,0.404836,0.311752,0.329173
4,0.428497,0.427457,0.277951,0.291212


## For class accuracy

In [ ]:
def extract_class_final_accuracy(s):
    data = parse_run_string(s)  # list of class curves
    
    return [float(class_curve[-1][1]) for class_curve in data]

def extract_class_auc(s):
    data = parse_run_string(s)
    
    return [
        sum(float(acc) for _, acc in class_curve) / len(class_curve)
        for class_curve in data
    ]

In [33]:
def clean_class_data(df):
    rows = []

    for run_idx in range(len(df)):
        row = df.iloc[run_idx]

        random = extract_class_final_accuracy(row["random"])
        margin = extract_class_final_accuracy(row["margin"])
        least = extract_class_final_accuracy(row["least_confident"])
        entropy = extract_class_final_accuracy(row["entropy"])

        n_classes = len(random)

        for c in range(n_classes):
            rows.append({
                "run": run_idx,
                "class": c,
                "random": random[c],
                "margin": margin[c],
                "least_conf": least[c],
                "entropy": entropy[c],
            })

    return pd.DataFrame(rows)

In [34]:
df_ec = clean_class_data(pd.read_csv("results/even_class_accuracy.csv"))
df_uc = clean_class_data(pd.read_csv("results/uneven_class_accuracy.csv"))

In [37]:
df_ec.head(10)

,run,class,random,margin,least_conf,entropy
0,0,0,0.087432,0.571949,0.446266,0.132969
1,0,1,0.522769,0.167577,0.522769,0.459016
2,0,2,0.420765,0.406193,0.504554,0.193078
3,0,3,0.630909,0.801818,0.000000,0.000000
4,0,4,0.460000,0.576364,0.120000,0.463636
5,0,5,0.076503,0.134791,0.296903,0.551913
6,0,6,0.961818,0.263636,0.060000,0.010909
7,1,0,0.571949,0.409836,0.147541,0.120219
8,1,1,0.283636,0.203636,0.547273,0.581818
9,1,2,0.385455,0.323636,0.721818,0.809091


## Statistical tests